# 伝達AI RQ1 実験 Notebook

最終更新: 2026-08-07 00:00 JST

このNotebookは、研究質問RQ1をColab上で確認するための実行用Notebookです。

RQ1: 観察可能な学習ログから、学習者の理解度、弱点スキル、誤概念、行動特徴をどの程度正確に推定できるか。

このNotebookでは、生徒AIの内部状態を伝達AIに渡しません。内部状態は評価用のGround Truthとしてのみ使います。


## 0. Colabで使うときの流れ

1. GitHubから最新版を取得する
2. 依存関係を入れる
3. RQ1実験を実行する
4. 比較表と共有用txtを確認する

LLMロードは行いません。まずはmock modelとルールベース伝達AIで、研究評価の流れが成立するかを確認します。


## Colabセットアップセル

Colabでは最初にこのセルを実行してください。GitHubの最新版取得、依存関係の導入、`src` のimport準備、出力先確認まで行います。


In [ ]:
# Colab setup: clone/update repository, install dependencies, prepare imports

import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_ROOT = Path("/content/student-ai")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print("Cloning repository...")
        !git clone {REPO_URL} /content/student-ai
    os.chdir(PROJECT_ROOT)
    print("Updating repository...")
    !git fetch origin main
    !git reset --hard origin/main
    !git log -1 --oneline
    print("Installing dependencies...")
    !pip install -q -r requirements.txt
else:
    PROJECT_ROOT = Path.cwd()
    os.chdir(PROJECT_ROOT)
    print("Local environment detected. Repository update and pip install are skipped.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for path in ["data/assessments", "data/observations", "data/teacher_beliefs"]:
    Path(path).mkdir(parents=True, exist_ok=True)

print("project_root:", PROJECT_ROOT)
print("in_colab:", IN_COLAB)
print("src import path ready:", (PROJECT_ROOT / "src").exists())
print("output_dir ready:", Path("data/assessments").resolve())


## 2. 実験条件を設定する

まずは軽量条件で動かします。

- `CLASS_SIZE`: 生徒数
- `QUESTION_COUNT`: 使用する問題数
- `SEED`: 再現性確認用

大きくしたい場合は `CLASS_SIZE=10`, `QUESTION_COUNT=20` などに変更できます。


In [ ]:
CLASS_ID = "class_10_mixed"
CLASS_SIZE = 5
QUESTION_COUNT = 6
SEED = 20260807

print({
    "CLASS_ID": CLASS_ID,
    "CLASS_SIZE": CLASS_SIZE,
    "QUESTION_COUNT": QUESTION_COUNT,
    "SEED": SEED,
})


## 3. RQ1実験を実行する

このセルで以下をまとめて実行します。

```text
Ground Truth Stateを読み込む
  -> Observable Eventを生成
  -> 複数の伝達方式でTeacher Belief Stateを推定
  -> Ground Truthと比較
  -> JSON/txtを保存
```


In [ ]:
from pprint import pprint

from src.experiment import (
    export_rq1_communication_ai_results,
    run_rq1_communication_ai_experiment,
)

rq1_result = run_rq1_communication_ai_experiment(
    class_id=CLASS_ID,
    class_size=CLASS_SIZE,
    question_count=QUESTION_COUNT,
    seed=SEED,
)

rq1_outputs = export_rq1_communication_ai_results(rq1_result)

print("outputs:")
pprint(rq1_outputs)
print("
summary:")
pprint(rq1_result["summary"])
print("
leakage_check:")
pprint(rq1_result["leakage_check"])


## 4. 比較表を見る

伝達方式と観察情報アブレーションごとの結果を表で確認します。

見るポイント:

- `skill_mae`: スキル習熟度の誤差。低いほどよい
- `overall_mae`: 全体理解度の誤差。低いほどよい
- `misconception_f1`: 誤概念検出。高いほどよい
- `trait_accuracy`: 個人特徴推定。高いほどよい
- `brier`, `ece`: 確信度の較正。低いほどよい


In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(rq1_result["comparison_rows"])
display(comparison_df)


## 5. 観察情報アブレーションを可視化する

発話や回答内容を入れることで、どの指標が変わるかを確認します。


In [ ]:
import matplotlib.pyplot as plt

plot_df = comparison_df.copy()
methods = plot_df["method"].unique()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for method in methods:
    subset = plot_df[plot_df["method"] == method]
    axes[0].plot(subset["ablation_condition"], subset["skill_mae"], marker="o", label=method)
    axes[1].plot(subset["ablation_condition"], subset["trait_accuracy"], marker="o", label=method)
    axes[2].plot(subset["ablation_condition"], subset["misconception_f1"], marker="o", label=method)

axes[0].set_title("Skill MAE")
axes[0].set_ylabel("lower is better")
axes[1].set_title("Trait Accuracy")
axes[1].set_ylabel("higher is better")
axes[2].set_title("Misconception F1")
axes[2].set_ylabel("higher is better")

for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, alpha=0.3)

axes[2].legend(loc="best")
plt.tight_layout()
plt.show()


## 6. Teacher Belief Stateの中身を見る

拡張伝達AIでは、推定値だけでなく、根拠イベント、反証イベント、観察数、情報不足、追加観察候補を持ちます。


In [ ]:
# enhanced_communication_ai / all_observable の最初の生徒を確認
selected_eval = next(
    item for item in rq1_result["evaluations"]
    if item["method"] == "enhanced_communication_ai" and item["ablation_condition"] == "all_observable"
)

first_student_id = sorted(selected_eval["teacher_beliefs"].keys())[0]
print("student_id:", first_student_id)
pprint(selected_eval["teacher_beliefs"][first_student_id])


## 7. 失敗例を見る

推定が外れた例を確認します。論文では、この失敗例が「どの観察が不足していたか」を説明する材料になります。


In [ ]:
for item in rq1_result["evaluations"]:
    if item["method"] == "enhanced_communication_ai" and item["ablation_condition"] == "all_observable":
        print(item["method"], "/", item["ablation_condition"])
        pprint(item["failure_examples"][:3])
        break


## 8. Codex/ChatGPT共有用txtを表示する

出力を貼る量が多いときは、このtxtファイルをそのまま添付してください。


In [ ]:
from pathlib import Path

summary_path = Path(rq1_outputs["txt"])
print(summary_path)
print(summary_path.read_text(encoding="utf-8")[:5000])


## 9. 実験の意味

このNotebookで示せること:

- 生徒AIの内部状態を直接見ずに、観察ログだけからTeacher Beliefを推定できる
- 伝達方式ごとの差を同じデータで比較できる
- 観察情報を削ったときの性能変化を確認できる
- 失敗例と追加観察候補を出せる

このNotebookだけでは主張しないこと:

- 実際の人間生徒の心理状態を正確に推定できること
- 実教室で同じ性能が出ること
- 教師AIが人間教師より優れていること
